# RAGAS: Retrieve vs AgenticRetrieveStream on a Bedrock Managed KB

This notebook scores **two retrieval methods head to head** with RAGAS and shows where one beats the other. Both run against Bedrock Managed Knowledge Bases, so both use only the APIs a Managed KB supports.

| Arm | Retrieval method | Reaches | Generates the answer |
|-----|------------------|---------|----------------------|
| **A (baseline)** | `Retrieve` | Exactly **one** KB (API takes a single `knowledgeBaseId`) | No, so we add a separate `Converse` call |
| **B** | `AgenticRetrieveStream` | **Multiple** KBs in one call (the docs cite up to 5), and it decomposes the query | Yes, built in, with `generateResponse=True` |

**The thesis.** `Retrieve` does one vector search against one knowledge base. `AgenticRetrieveStream` can register several KBs *and* it decomposes a question into sub-queries, retrieving for each. That buys it two advantages this notebook measures:

1. **Multi-KB reach**: when an answer's evidence is split across two KBs, `Retrieve` can reach only one of them, so it gathers at most half the evidence.
2. **Query decomposition**: even within a single KB, a multi-part question ("cash *and* the three risks") is answered better when it is split into focused sub-queries than by one broad search.

So Arm B wins clearly on cross-KB questions, and often edges ahead on multi-part single-KB questions too. On a simple, single-fact question the two are expected to tie. We hold everything else constant and let RAGAS score the difference.

> Managed KBs support only `Retrieve` and `AgenticRetrieveStream`. `RetrieveAndGenerate` is **not** supported for Managed KBs, which is why Arm A generates its answer with a separate `Converse` call rather than `RetrieveAndGenerate`.

### How this differs from the other eval notebooks here

| Notebook | How it works | Works for a Managed KB? |
|----------|--------------|-------------------------|
| `02-bedrock-evaluation-job.ipynb` | Bedrock Evaluation Job (`CreateEvaluationJob`) | Retrieval only |
| `03-agentcore-evaluation-for-managed-kb.ipynb` | AgentCore Evaluations (scores OTEL trace spans) | Yes |
| This notebook | You retrieve and generate answers yourself with two methods, then score both with RAGAS | Yes |


## Prerequisites

- AWS credentials in a **us**, **eu**, or **ap** region (Managed KB and the Claude and Titan inference profiles live there).
- Model access turned on for the generation, judge, and embedding models below.
- IAM permissions: `bedrock:Retrieve`, `bedrock:AgenticRetrieveStream`, `bedrock:GetDocumentContent`, `bedrock:InvokeModel*`, `bedrock:InvokeModelWithResponseStream`, plus the KB/S3/IAM permissions used by `utils/managed_knowledge_base.py`.
- Two Managed KBs: a **finance** KB (Octank Financial 10-K) and a **weather** KB (NOAA/CRS tornado report). This notebook can create them for you, or you can pass in the IDs of KBs you already have.
- `ragas` and `datasets` (installed below), plus this repo's `requirements.txt`.
- Kernel: pick `Python 3`.


In [ ]:
%pip install --upgrade pip --quiet
# RAGAS 0.1.21 needs the older langchain stack (langchain-core below 0.3), so pin it.
# We also pin langchain-openai and openai: ragas 0.1.21 imports langchain_openai at
# import time, and an unpinned version can drag in langchain-core >= 0.3 and break
# the pin above.
%pip install --quiet \
    "langchain-core==0.2.43" \
    "langchain==0.2.16" \
    "langchain-community==0.2.17" \
    "langchain-text-splitters==0.2.4" \
    "langchain-aws==0.1.18" \
    "langchain-openai==0.1.25" \
    "openai>1,<2" \
    "ragas==0.1.21" \
    "datasets>=2.14.0"
# AgenticRetrieveStream needs boto3 1.43 or newer. langchain-aws pins boto3 below
# 1.35, but that pin does not matter here because we pass in our own client, so
# bump boto3 back up afterward. Expect a pip dependency-conflict warning; it is
# harmless as long as we always pass client=bedrock_runtime into the langchain wrappers.
%pip install --upgrade --quiet "boto3>=1.43.0" "botocore>=1.43.0" "s3transfer>=0.13.0" 

In [ ]:
# Restart the kernel so the re-pinned langchain-core loads before `import ragas`.
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Papermill parameters.
# This cell is tagged 'parameters' so papermill can set these for a headless run.
#
# KB reuse vs create:
#   - Set kb_finance_id AND kb_weather_id to reuse two KBs you already have.
#   - Set only kb_id to run in single-KB mode (cross-KB questions are dropped).
#   - Leave all three as None with create_kbs=True to have this notebook build them.
kb_finance_id = None      # finance KB (Octank 10-K). None -> create one if create_kbs
kb_weather_id = None      # weather KB (tornado report). None -> create one if create_kbs
kb_id         = None      # single-KB fallback: only this set -> degraded single-KB mode
kb_domain     = 'finance' # which content the single kb_id holds: 'finance' or 'weather'
create_kbs    = True      # if False and no ids are given, stop instead of provisioning

# Retrieval depth. The two APIs have DIFFERENT limits, so they are separate knobs:
#   - Retrieve (Arm A): numberOfResults accepts 1-100.
#   - AgenticRetrieveStream (Arm B): maxNumberOfResults accepts 1-10 (per retriever).
# Keep them equal for a fair comparison; they only differ if you deliberately raise one.
retrieve_num_results = 5  # Arm A chunks per query (1-100)
agentic_max_results  = 5  # Arm B chunks per KB    (1-10)
max_questions = 9         # cap on the eval set (there are 9 questions)
robustness_n  = 10        # Arm A robustness re-run depth (Step 10), 1-100
fail_on_error = True      # stop the run on any hole in the data: a generation error, a
                          # dropped row, a failed robustness re-run, or a NaN RAGAS score.
                          # Keeps a reduced/partial sample from reading as a clean pass.
                          # Set False for a best-effort run (scores the shared usable subset).
cleanup       = False     # if True, delete ONLY the resources this notebook created

## Step 1: Configuration

Pick the region, build the model IDs and ARNs, and create the boto3 clients. The **same generation model** is used by both arms and the **same judge and embedding models** score both arms, so the only thing that differs between the arms is the retrieval method.

In [ ]:
import boto3
import json
import time
import sys
from botocore.config import Config

sys.path.insert(0, "../..")

session = boto3.session.Session()
region = session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

# Longer timeouts: agentic retrieval plus generation can take a while.
runtime_config = Config(connect_timeout=120, read_timeout=120, retries={'max_attempts': 3})
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime', region_name=region, config=runtime_config)
bedrock_runtime = boto3.client('bedrock-runtime', region_name=region, config=runtime_config)
s3_client = boto3.client('s3', region_name=region)

# Validate the retrieval-depth knobs now, so a bad value fails here with a clear
# message instead of a confusing per-question error later. Use `type(x) is int`, not
# isinstance: in Python `True`/`False` ARE ints, so isinstance(True, int) passes, and
# Bedrock then rejects the bool with a SerializationException. Reject it up front.
def _check_int(name, value, lo, hi):
    if type(value) is not int or not (lo <= value <= hi):
        raise ValueError(f'{name} must be an integer from {lo} to {hi}, got {value!r}')

# Arm A (Retrieve) allows 1-100; Arm B (AgenticRetrieveStream) allows only 1-10.
_check_int('retrieve_num_results', retrieve_num_results, 1, 100)
_check_int('agentic_max_results', agentic_max_results, 1, 10)
_check_int('robustness_n', robustness_n, 1, 100)

# ── Models ────────────────────────────────────────────────────────────
# Pick the inference-profile prefix for the current region. Stop early on an
# unsupported region instead of building a mismatched ARN (e.g. a 'us.' profile in
# ca-central-1) that would only fail later inside the retrieval calls.
region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region and region.startswith(k)), None)
if cris_prefix is None:
    raise ValueError(
        f'Region {region!r} has no matching inference-profile prefix. '
        'This notebook expects a us, eu, or ap region, where Managed KB and the '
        'Claude and Titan inference profiles are available. Switch regions, or '
        'edit region_prefix_map and the model IDs for your region.'
    )

# Generation model. AgenticRetrieveStream needs the full inference-profile ARN;
# Converse (Arm A) takes the bare inference-profile id. Both point at the SAME model,
# which keeps the comparison fair.
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'
generation_model_id  = f'{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

# The judge model RAGAS uses (same family as generation), and the embedding model.
judge_model_id = f'{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'
embedding_model_id = 'amazon.titan-embed-text-v2:0'

def dedup_contexts(chunks):
    # Drop repeated chunks, comparing on whitespace-normalized text. Applied to BOTH
    # arms so the comparison measures distinct retrieved evidence: Retrieve can return
    # the same passage twice, and an agentic run can surface the same chunk from more
    # than one sub-query. Deduping one arm but not the other would bias
    # context_precision. Order is preserved.
    seen, out = set(), []
    for t in chunks:
        key = ' '.join((t or '').split())
        if key and key not in seen:
            seen.add(key)
            out.append(t)
    return out

print(f'Region:      {region}')
print(f'Account:     {account_id}')
print(f'Generation:  {generation_model_arn}')
print(f'Judge:       {judge_model_id}')
print(f'Embeddings:  {embedding_model_id}')

## Step 2: Get two Knowledge Bases

The comparison needs two KBs holding **different** documents:

- **Finance KB**: Octank Financial's 10-K (`synthetic_dataset/octank_financial_10K.pdf`)
- **Weather KB**: a NOAA/CRS tornado report (`synthetic_dataset/tornadoes_report.pdf`)

The cell below picks one of three paths:

1. **Reuse**: you passed `kb_finance_id` and `kb_weather_id`. Nothing is created.
2. **Single-KB (degraded)**: you passed only `kb_id`, and `kb_domain` says whether it holds `'finance'` or `'weather'` content. The notebook runs only that domain's questions and drops the rest (cross-KB questions need both KBs, and the other domain's questions have no KB to hit). The multi-KB advantage can't show, so the arms tie on simple questions, though Arm B can still edge ahead on multi-part ones via query decomposition. This keeps the notebook runnable when you only have one KB.
3. **Create**: no IDs and `create_kbs=True`. The notebook uploads both PDFs to one bucket (under `financial/` and `weather/` prefixes) and builds a KB for each. This takes a few minutes.

In [ ]:
two_kb_mode = True
kb_fin = kb_wea = None         # ManagedKnowledgeBase objects, only set when we create
# Cleanup state. We only ever delete what THIS notebook created, and we track each
# piece separately so a failure partway through still leaves a correct record of
# what exists (a single 'both KBs done' flag would leak resources on partial failure).
created = {'bucket': None, 'kb_fin': False, 'kb_wea': False, 'keys': []}

# The comparison is meaningless if both slots point at the same KB (the "two" retrievers
# would query one store), so reject that up front.
if kb_finance_id and kb_weather_id and kb_finance_id == kb_weather_id:
    raise ValueError(
        f'kb_finance_id and kb_weather_id are the same KB ({kb_finance_id}). Provide two '
        'different KBs (finance and weather), or pass just kb_id for single-KB mode.')

if kb_finance_id and kb_weather_id:
    print('Reusing the two KBs you passed in. Nothing created.')
    print(f'  finance KB: {kb_finance_id}')
    print(f'  weather KB: {kb_weather_id}')

elif kb_id and not (kb_finance_id or kb_weather_id):
    # One KB provided. kb_domain says whether it holds finance or weather content, so
    # we run that domain's questions against it. Cross-KB questions are dropped; Arm B
    # uses a single retriever.
    if kb_domain not in ('finance', 'weather'):
        raise ValueError(f"kb_domain must be 'finance' or 'weather', got {kb_domain!r}")
    if kb_domain == 'finance':
        kb_finance_id, kb_weather_id = kb_id, None
    else:
        kb_finance_id, kb_weather_id = None, kb_id
    two_kb_mode = False
    print(f'Single KB provided ({kb_domain}) -> degraded single-KB mode.')
    print(f'  Only {kb_domain} questions run; cross-KB questions are dropped and Arm B')
    print('  uses one retriever. Provide two KBs for the full multi-KB comparison.')

elif create_kbs:
    from utils.managed_knowledge_base import ManagedKnowledgeBase

    suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]
    bucket_name = f'bedrock-ragas-eval-{suffix}-{account_id}'

    # Create the bucket. Do NOT reuse a pre-existing one: cleanup would later empty and
    # delete the whole bucket, so an accidental name collision could destroy someone
    # else's data. A collision on this timestamped, account-scoped name is very
    # unlikely; if it happens we want the error, not a silent reuse.
    try:
        if region == 'us-east-1':
            s3_client.create_bucket(Bucket=bucket_name)
        else:
            s3_client.create_bucket(Bucket=bucket_name,
                                    CreateBucketConfiguration={'LocationConstraint': region})
        created['bucket'] = bucket_name   # record now: we own this bucket and may delete it
    except s3_client.exceptions.BucketAlreadyExists:
        raise RuntimeError(
            f'Bucket {bucket_name} already exists and is owned by another account. '
            'Re-run to get a new timestamped name.')
    except s3_client.exceptions.BucketAlreadyOwnedByYou:
        raise RuntimeError(
            f'Bucket {bucket_name} already exists in this account. This notebook only '
            'deletes buckets it created, so it will not reuse or delete a pre-existing '
            'one. Re-run to get a new timestamped name, or delete that bucket yourself.')
    print(f'Created bucket: {bucket_name}')

    for key, path in [('financial/octank_financial_10K.pdf', '../../synthetic_dataset/octank_financial_10K.pdf'),
                      ('weather/tornadoes_report.pdf', '../../synthetic_dataset/tornadoes_report.pdf')]:
        s3_client.upload_file(path, bucket_name, key)
        created['keys'].append(key)
    print('Uploaded both PDFs.')

    print('\nCreating finance KB ...')
    kb_fin = ManagedKnowledgeBase(
        kb_name=f'ragas-eval-finance-{suffix}',
        data_sources=[{'type': 'S3', 'bucket_name': bucket_name, 's3_prefix': 'financial/'}],
        embedding_model=None,            # managed default (no extra cost)
        kb_description='RAGAS eval: Octank Financial 10-K',
        region_name=region, suffix=f'evalfin{suffix}',
    )
    created['kb_fin'] = True
    kb_fin.start_ingestion_job()         # blocks and polls to COMPLETE

    print('\nCreating weather KB ...')
    kb_wea = ManagedKnowledgeBase(
        kb_name=f'ragas-eval-weather-{suffix}',
        data_sources=[{'type': 'S3', 'bucket_name': bucket_name, 's3_prefix': 'weather/'}],
        embedding_model=None,
        kb_description='RAGAS eval: NOAA/CRS tornado report',
        region_name=region, suffix=f'evalwea{suffix}',
    )
    created['kb_wea'] = True
    kb_wea.start_ingestion_job()

    kb_finance_id, kb_weather_id = kb_fin.kb_id, kb_wea.kb_id
    print(f'\nfinance KB: {kb_finance_id}')
    print(f'weather KB: {kb_weather_id}')

else:
    raise ValueError(
        'No KB IDs provided and create_kbs=False. Either set kb_finance_id and '
        'kb_weather_id (or kb_id), or set create_kbs=True to build them.'
    )

# Save for other notebooks / a later cleanup run.
%store kb_finance_id
%store kb_weather_id
print(f'\ntwo_kb_mode = {two_kb_mode}')

## Step 3: The evaluation dataset

Nine questions in three groups. RAGAS needs a `question` and a `ground_truth` (reference answer) for each; `context_precision`, `context_recall`, and `answer_correctness` are all scored against that ground truth, so the reference answers are written to match the ingested PDFs exactly.

| Group | Count | Answerable from | What we expect |
|-------|-------|-----------------|----------------|
| `finance-only` | 3 | Finance KB alone | Close; Arm B can edge ahead on multi-part questions |
| `tornado-only` | 3 | Weather KB alone | Roughly a tie |
| `cross-kb` | 3 | **Both** KBs | Arm B **wins clearly** |

Each row also carries an `oracle_kb` label. For Arm A (single-KB `Retrieve`) we let the baseline point at the *correct* KB for single-KB questions, and at the KB holding the *larger* share of the answer for cross-KB questions. That is a more generous baseline than a real user gets, which makes the point stronger: even an oracle-routed single-KB baseline can't answer a question whose evidence is split across two KBs.

The single-KB groups are the control. A simple single-fact question should tie; a multi-part one may still favor Arm B because it decomposes the question. The questions and labels are fixed **before** running so scores can't be cherry-picked afterward.

In [ ]:
EVAL_SET = [
    # ── finance-only (oracle_kb='finance'): single-KB, may tie ────────
    {
        'question': "What was Octank Financial's total revenue in 2021, how did it change from 2020, and what drove the change?",
        'ground_truth': "Octank Financial reported total revenue of $5,000 million in 2021, an increase of $500 million (about 11.1%) from $4,500 million in 2020. The growth was driven by strong performance in the company's core business segments and the acquisition of several smaller companies.",
        'category': 'finance-only', 'oracle_kb': 'finance',
    },
    {
        'question': "What was the fair value of Octank Financial's investment portfolio at year-end 2021, how much did it grow, and how is the portfolio classified?",
        'ground_truth': "As of December 31, 2021, Octank Financial's investment portfolio had a fair value of $5.2 billion, a 12% increase from the prior year. It is classified into three categories: available-for-sale (AFS), held-to-maturity (HTM), and trading securities.",
        'category': 'finance-only', 'oracle_kb': 'finance',
    },
    {
        'question': "How much cash and cash equivalents did Octank Financial hold at year-end 2021 versus 2020, and what are the three main risks to its investment portfolio?",
        'ground_truth': "Octank held cash and cash equivalents of $12,567,000 as of December 31, 2021, up from $9,854,000 a year earlier, an increase of 27.5%. Its investment portfolio is subject to three main risks: market risk (value falling due to market-price changes), credit risk (an issuer defaulting), and liquidity risk (being unable to sell securities in a timely manner or at fair value). The company manages these by diversifying the portfolio, monitoring issuer creditworthiness, and regularly assessing the portfolio.",
        'category': 'finance-only', 'oracle_kb': 'finance',
    },
    # ── tornado-only (oracle_kb='weather'): single-KB, may tie ────────
    {
        'question': "On the EF-scale, what 3-second gust wind-speed range corresponds to each EF number from EF0 to EF5, and how is a tornado's strength estimated?",
        'ground_truth': "The Enhanced Fujita (EF) scale, adopted in 2007, estimates a tornado's strength from the damage it caused (using 28 damage indicators) rather than from measured winds. The 3-second gust ranges are: EF0 65-85 mph, EF1 86-110 mph, EF2 111-135 mph, EF3 136-165 mph, EF4 166-200 mph, and EF5 over 200 mph.",
        'category': 'tornado-only', 'oracle_kb': 'weather',
    },
    {
        'question': "What is the difference between a tornado watch and a tornado warning, and who issues each?",
        'ground_truth': "A tornado watch is issued by NOAA/NWS's Storm Prediction Center (SPC) when conditions become favorable for multiple tornadoes or a single intense tornado; it typically lasts six to eight hours, and SPC aims to issue it at least two hours before the first tornado. A tornado warning is issued by local Weather Forecast Offices (WFOs) when a tornado has been sighted or indicated by weather radar; it includes the areas at risk, time frames, specific hazards, and recommended safety precautions.",
        'category': 'tornado-only', 'oracle_kb': 'weather',
    },
    {
        'question': "About how many tornadoes does the United States report each year, and in which three regions do they form most frequently?",
        'ground_truth': "The United States reports approximately 1,200 tornadoes per year, based on official data going back to the 1950s. They form most frequently in three regions: the southern plains (e.g., Texas, Oklahoma, Kansas), the Gulf Coast (e.g., Alabama, Florida, Louisiana, Mississippi), and the northern plains and upper Midwest (e.g., North and South Dakota, Nebraska, Iowa, Minnesota).",
        'category': 'tornado-only', 'oracle_kb': 'weather',
    },
    # ── cross-kb: needs a finance fact AND a tornado fact, Arm B wins ─
    {
        'question': "Octank Financial's 10-K names natural disasters among the market-risk factors that can hurt its investments. Using the tornado report, roughly how many tornadoes does the US see per year and which regions are most exposed?",
        'ground_truth': "Octank Financial's 10-K identifies natural disasters, along with changes in interest rates, inflation, and economic conditions, as factors that can negatively affect its investments. The tornado report states the United States averages about 1,200 tornadoes per year (official data since the 1950s), forming most frequently in the southern plains (Texas, Oklahoma, Kansas), the Gulf Coast (Alabama, Florida, Louisiana, Mississippi), and the northern plains and upper Midwest (the Dakotas, Nebraska, Iowa, Minnesota).",
        'category': 'cross-kb', 'oracle_kb': 'finance',
    },
    {
        'question': "Compare how each source classifies severity: what are the three risk categories Octank Financial uses for its investment portfolio, versus how NOAA's EF-scale classifies tornado severity?",
        'ground_truth': "Octank Financial classifies its investment-portfolio risk into three categories: market risk, credit risk, and liquidity risk. NOAA's Enhanced Fujita (EF) scale classifies tornado severity from EF0 to EF5 by estimated 3-second gust wind speed: EF0 65-85 mph, EF1 86-110, EF2 111-135, EF3 136-165, EF4 166-200, and EF5 over 200 mph, estimated from 28 damage indicators rather than measured winds.",
        'category': 'cross-kb', 'oracle_kb': 'finance',
    },
    {
        'question': "Prepare a two-part brief: (a) Octank Financial's cash and cash equivalents at year-end 2021 and their year-over-year change, and (b) the difference between a tornado watch and a tornado warning so the team knows when to act.",
        'ground_truth': "(a) Octank Financial held cash and cash equivalents of $12,567,000 as of December 31, 2021, up from $9,854,000 the prior year, an increase of 27.5%. (b) A tornado watch, issued by NOAA/NWS's Storm Prediction Center, means conditions are favorable for tornadoes (typically lasting six to eight hours and aimed to be issued at least two hours ahead), whereas a tornado warning, issued by local Weather Forecast Offices, means a tornado has been sighted or indicated by radar and immediate action is needed.",
        'category': 'cross-kb', 'oracle_kb': 'weather',
    },
]

# Keep only questions this setup can actually answer.
#   - cross-KB questions need BOTH KBs, so drop them in single-KB mode.
#   - every remaining question is routed (by oracle_kb) to one KB for Arm A; drop any
#     whose KB is not available. In single-KB mode the other domain's KB is None, so
#     this is what removes that domain's rows (otherwise Arm A would call Retrieve with
#     knowledgeBaseId=None and crash).
_available = {'finance': kb_finance_id, 'weather': kb_weather_id}
before = len(EVAL_SET)
if not two_kb_mode:
    EVAL_SET = [r for r in EVAL_SET if r['category'] != 'cross-kb']
EVAL_SET = [r for r in EVAL_SET if _available.get(r['oracle_kb'])]
dropped = before - len(EVAL_SET)
if dropped:
    print(f'Dropped {dropped} question(s) this setup cannot answer '
          '(cross-KB in single-KB mode, or questions routed to a KB you did not provide).')
if not EVAL_SET:
    raise ValueError(
        'No answerable questions left. In single-KB mode only the questions matching '
        "kb_domain run (kb_domain='finance' -> finance questions, 'weather' -> tornado "
        'questions). Pass both kb_finance_id and kb_weather_id for the full comparison.'
    )

# Trim to max_questions. type(...) is int rejects booleans (True is an int in Python).
if type(max_questions) is not int or max_questions < 1:
    raise ValueError(f'max_questions must be a positive integer, got {max_questions!r}')
EVAL_SET = EVAL_SET[:max_questions]

from collections import Counter
print(f'Evaluating {len(EVAL_SET)} questions:', dict(Counter(r['category'] for r in EVAL_SET)))

## Step 4: Arm A uses `Retrieve` (one KB), then `Converse`

`Retrieve` returns chunks only; it does not generate. So Arm A is two steps:

1. `Retrieve` from the **one** KB the `oracle_kb` label points at (`managedSearchConfiguration.numberOfResults = retrieve_num_results`).
2. `Converse` with those chunks stuffed into a strict, grounded RAG prompt to produce the answer.

Both the retrieved chunks and the generated answer go to RAGAS. The generation model is the same as Arm B, and `retrieve_num_results` is set equal to `agentic_max_results`, so the retrieval method is the only thing that differs.

In [ ]:
ORACLE = {'finance': kb_finance_id, 'weather': kb_weather_id}

RAG_PROMPT = (
    "You are a precise assistant. Answer the QUESTION using ONLY the CONTEXT below. "
    "If the context does not contain the answer, say you cannot answer from the "
    "provided documents. Do not use outside knowledge. Be concise and factual.\n\n"
    "CONTEXT:\n{context}\n\nQUESTION: {question}\n\nANSWER:"
)

def arm_a_retrieve(question, oracle_kb_id, n):
    """Retrieve chunks from ONE KB. Retrieve is single-KB by API contract.

    We take a single page of up to n results. Retrieve also returns a nextToken for
    paging, but we deliberately do not follow it: numberOfResults=n already caps the
    budget, and that cap is the control we hold equal to Arm B. Following nextToken
    would pull extra context and break the comparison.
    """
    resp = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=oracle_kb_id,
        retrievalQuery={'text': question},
        retrievalConfiguration={
            # Managed KB -> managedSearchConfiguration (NOT vectorSearchConfiguration).
            'managedSearchConfiguration': {'numberOfResults': n}
        },
    )
    results = resp.get('retrievalResults', [])
    chunks = [r['content']['text'] for r in results if r.get('content', {}).get('text')]
    return dedup_contexts(chunks)   # dedup both arms identically (see dedup_contexts)

def arm_a_answer_and_contexts(question, oracle_kb_id, n=None):
    # `n is None` (not `n or ...`): a caller passing n=0 should be an explicit value,
    # not silently replaced by the default. (n=0 would fail the API, which is correct.)
    n = retrieve_num_results if n is None else n
    contexts = arm_a_retrieve(question, oracle_kb_id, n)
    prompt = RAG_PROMPT.format(context='\n\n---\n\n'.join(contexts), question=question)
    resp = bedrock_runtime.converse(
        modelId=generation_model_id,   # Converse takes the bare inference-profile id
        messages=[{'role': 'user', 'content': [{'text': prompt}]}],  # content is a LIST here
        inferenceConfig={'maxTokens': 1024, 'temperature': 0.0},
    )
    answer = resp['output']['message']['content'][0]['text']
    return answer, contexts

# We catch per-question errors so one bad question does not abort the whole run, but we
# COUNT them and surface the count. A run with failures is not a clean run: empty answers
# get dropped before scoring, so a silent failure would quietly shrink the sample.
arm_a_answers, arm_a_contexts, arm_a_errors = [], [], []
for i, row in enumerate(EVAL_SET, 1):
    try:
        ans, ctx = arm_a_answer_and_contexts(row['question'], ORACLE[row['oracle_kb']])
    except Exception as e:
        print(f'  [{i}/{len(EVAL_SET)}] ERROR: {type(e).__name__}: {e}')
        arm_a_errors.append((i, repr(e)))
        ans, ctx = '', []
    arm_a_answers.append(ans)
    arm_a_contexts.append(ctx)
    print(f"  [{i}/{len(EVAL_SET)}] {row['category']:12s} answer chars={len(ans):4d}, chunks={len(ctx)}")
    time.sleep(2)  # small pause so we do not get throttled

print(f'\nArm A generated {sum(1 for a in arm_a_answers if a)} / {len(EVAL_SET)} answers')
if arm_a_errors:
    print(f'WARNING: {len(arm_a_errors)} Arm A question(s) failed and were left empty. '
          'Step 8 will stop (fail_on_error=True) rather than score a reduced sample.')

## Step 5: Arm B uses `AgenticRetrieveStream` over both KBs

One call registers **both** KBs as retrievers. The model plans the query, routes each part to the KB that can answer it, retrieves (up to `agentic_max_results` per KB, and this API caps that at 10), and generates the answer with `generateResponse=True`. We keep the answer, the chunks (de-duplicated by text, since the same passage can surface from more than one sub-query), and the trace events (used in Step 6).

Each retriever gets a short `description`; the model uses those to route sub-queries, so they matter for multi-KB routing.

In [ ]:
FIN_DESC = "Octank Financial 10-K: revenue, cash and investments, and market, credit, and liquidity risk factors."
WEA_DESC = "NOAA/CRS tornado report: EF-scale wind-speed bands, tornado watch vs warning, annual counts, and high-frequency regions."

# One retriever per KB that actually exists. Build from whichever ids are set, so this
# works whether single-KB mode holds the finance OR the weather KB. (A hardcoded finance
# entry would send knowledgeBaseId=None in weather mode and every Arm B call would fail.)
# The docs cite a max of 5 retrievers per call; this notebook only ever uses 1 or 2.
RETRIEVERS = []
if kb_finance_id:
    RETRIEVERS.append((kb_finance_id, FIN_DESC))
if kb_weather_id:
    RETRIEVERS.append((kb_weather_id, WEA_DESC))
if not RETRIEVERS:
    raise ValueError('No knowledge base available for Arm B. Set kb_finance_id and/or kb_weather_id.')

def arm_b_answer_and_contexts(question, retriever_ids):
    retrievers = [
        {
            'configuration': {
                'knowledgeBase': {
                    'knowledgeBaseId': kbid,
                    # maxNumberOfResults is PER-RETRIEVER and capped at 10 by this API
                    # (Retrieve allows 100, AgenticRetrieveStream does not).
                    'retrievalOverrides': {'maxNumberOfResults': agentic_max_results},
                }
            },
            'description': desc,
        }
        for kbid, desc in retriever_ids
    ]
    resp = bedrock_agent_runtime.agentic_retrieve_stream(
        messages=[{'role': 'user', 'content': {'text': question}}],  # content is a DICT here
        retrievers=retrievers,
        agenticRetrieveConfiguration={
            'foundationModelConfiguration': {
                'bedrockFoundationModelConfiguration': {
                    'modelConfiguration': {'modelArn': generation_model_arn}  # inference-profile ARN
                },
                'type': 'BEDROCK_FOUNDATION_MODEL',
            },
            'foundationModelType': 'CUSTOM',
            'maxAgentIteration': 3,          # minimum allowed is 2
            'rerankingModelType': 'MANAGED', # free managed reranker for managed-embedding KBs
        },
        generateResponse=True,
    )

    # The response is a botocore EventStream: each event has exactly one key.
    # Errors can arrive INLINE mid-stream as *Exception events, not only as raised
    # exceptions, so we check for those too.
    results, gen_resp, text_chunks, traces, trace_failures = [], None, [], [], []
    for event in resp['stream']:
        if 'traceEvent' in event:
            traces.append(event['traceEvent'])
            attrs = event['traceEvent'].get('attributes', {})
            # A FAILED trace step, or any warnings/failures, means the agent hit trouble.
            if attrs.get('status') == 'FAILED' or attrs.get('failures') or attrs.get('warnings'):
                trace_failures.append(attrs)
        elif 'responseEvent' in event:
            text_chunks.append(event['responseEvent'].get('text', ''))
        elif 'result' in event:
            # ACCUMULATE across result events rather than overwriting: in practice one
            # terminal result event arrives, but a paged/multi-event stream would drop
            # data if we replaced. generatedResponse takes the last non-empty one.
            results.extend(event['result'].get('results', []))
            gen_resp = event['result'].get('generatedResponse') or gen_resp
        elif any(k.endswith('Exception') for k in event):
            raise RuntimeError(f'AgenticRetrieveStream error event: {event}')

    answer = (gen_resp or {}).get('answer') or ''.join(text_chunks)
    # Agentic result items carry content + metadata + sourceRetriever (no score/location).
    # Dedup with the same helper Arm A uses: the same passage can surface from more than
    # one sub-query, and deduping both arms identically keeps the comparison fair.
    contexts = dedup_contexts(r.get('content', {}).get('text', '') for r in results)
    return answer, contexts, traces, trace_failures

arm_b_answers, arm_b_contexts, arm_b_traces, arm_b_errors, arm_b_trace_warns = [], [], [], [], 0
for i, row in enumerate(EVAL_SET, 1):
    try:
        ans, ctx, tr, tf = arm_b_answer_and_contexts(row['question'], RETRIEVERS)
        if tf:
            arm_b_trace_warns += 1
            print(f"  [{i}/{len(EVAL_SET)}] trace reported {len(tf)} failure/warning step(s)")
    except Exception as e:
        print(f'  [{i}/{len(EVAL_SET)}] ERROR: {type(e).__name__}: {e}')
        arm_b_errors.append((i, repr(e)))
        ans, ctx, tr = '', [], []
    arm_b_answers.append(ans)
    arm_b_contexts.append(ctx)
    arm_b_traces.append(tr)
    print(f"  [{i}/{len(EVAL_SET)}] {row['category']:12s} answer chars={len(ans):4d}, chunks={len(ctx)}")
    time.sleep(2)

print(f'\nArm B generated {sum(1 for a in arm_b_answers if a)} / {len(EVAL_SET)} answers')
if arm_b_errors:
    print(f'WARNING: {len(arm_b_errors)} Arm B question(s) failed and were left empty. '
          'Step 8 will stop (fail_on_error=True) rather than score a reduced sample.')
if arm_b_trace_warns:
    print(f'NOTE: {arm_b_trace_warns} question(s) had trace failures/warnings (e.g. a KB '
          'that could not answer a sub-query, which is normal for cross-KB questions). '
          'Surfaced, not failed: a sub-query miss is expected when one KB lacks that '
          'half of the answer.')

## Step 6: Proof that Arm B decomposed the query and hit both KBs

For one cross-KB question, the trace below shows the planning step (sub-queries) and which retriever each sub-query hit. This is the mechanism behind Arm B's win: it reached both KBs and split the question across them, which a single `Retrieve` call cannot do.

In [ ]:
if two_kb_mode:
    idx = next((i for i, r in enumerate(EVAL_SET) if r['category'] == 'cross-kb'), None)
    if idx is None:
        print('No cross-KB question in the trimmed set.')
    else:
        kb_label = {kb_finance_id: 'finance', kb_weather_id: 'weather'}
        print(f'Question: {EVAL_SET[idx]["question"]}\n')
        for tr in arm_b_traces[idx]:
            attrs = tr.get('attributes', {})
            step, status = attrs.get('step', '?'), attrs.get('status', '')
            actions = attrs.get('actions', [])
            if step == 'Planning' or actions:
                print(f'[{step}] {status}')
            for action in actions:
                if 'retrieve' in action:
                    sub = action['retrieve'].get('inputQuery', {}).get('text', '')
                    hits = [s.get('identifier', '?') for s in action['retrieve'].get('sourceRetrievers', [])]
                    hits = [kb_label.get(h, h) for h in hits]
                    print(f'    sub-query -> {hits}: {sub}')
        print('\nArm B answer:\n', arm_b_answers[idx][:700])
else:
    print('Single-KB mode: no multi-KB trace to show.')

## Step 7: Set up RAGAS with Bedrock

RAGAS needs a judge model and an embedding model. We wrap two Bedrock models with `langchain-aws` and pass them in. We add **`answer_correctness`** to the usual four metrics because it compares the answer directly to the ground truth, which is where the cross-KB gap shows up most clearly.

We also lower `RunConfig(max_workers)`. The default of 16 fires too many concurrent judge calls at Bedrock and trips throttling, which RAGAS records as `NaN` scores rather than errors.

In [ ]:
from langchain_aws.chat_models.bedrock import ChatBedrock
from langchain_aws.embeddings.bedrock import BedrockEmbeddings
from datasets import Dataset
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness,
)

judge_llm = ChatBedrock(
    model_id=judge_model_id,
    client=bedrock_runtime,   # keep our boto3>=1.43 client (do not let langchain build one)
    model_kwargs={'max_tokens': 4096, 'temperature': 0.0},
)
embeddings = BedrockEmbeddings(model_id=embedding_model_id, client=bedrock_runtime)

# Tame Bedrock throttling: fewer concurrent judge calls, generous timeout/retries.
run_cfg = RunConfig(max_workers=4, timeout=180, max_retries=10)

METRICS = [faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness]
METRIC_NAMES = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall', 'answer_correctness']
print('RAGAS ready with', len(METRICS), 'metrics')

## Step 8: Score both arms on the same questions

`score_arm` builds a RAGAS dataset from an arm's answers and contexts, drops any row with no answer or no contexts (RAGAS can't score those), runs `evaluate`, and tags each row with its arm and category. RAGAS 0.1.21 expects the columns `question`, `answer`, `contexts` (a list of strings), and `ground_truth` (a single string).

In [ ]:
import pandas as pd

# Guard the sample. Generation errors, or (with fail_on_error) any row that would be
# dropped, stop the run rather than quietly scoring a smaller set that can still read
# as a clean pass.
_total_errors = len(arm_a_errors) + len(arm_b_errors)
if fail_on_error and _total_errors:
    raise RuntimeError(
        f'{_total_errors} question(s) errored during generation (Arm A: {len(arm_a_errors)}, '
        f'Arm B: {len(arm_b_errors)}). Stopping so a reduced sample is not scored as if '
        'complete. Fix the cause or set fail_on_error=False to score what succeeded.'
    )

# Score the SAME set of questions in both arms. A question is usable only if BOTH arms
# produced an answer and contexts for it; otherwise we would compare the arms on
# different question sets and bias the result. In best-effort mode (fail_on_error=False)
# we keep the shared usable subset; with fail_on_error the run has already stopped above
# if anything was missing.
def usable(answers, contexts):
    return [bool(a and a.strip() and c) for a, c in zip(answers, contexts)]

keep = [ua and ub for ua, ub in zip(usable(arm_a_answers, arm_a_contexts),
                                    usable(arm_b_answers, arm_b_contexts))]
n_drop = len(keep) - sum(keep)
if n_drop:
    print(f'Dropping {n_drop} question(s) not usable in BOTH arms so the comparison stays '
          'paired (best-effort mode).')
if fail_on_error and n_drop:
    raise RuntimeError(
        f'{n_drop} question(s) lacked an answer or contexts in at least one arm. Stopping '
        '(fail_on_error=True). Set it to False to score the shared usable subset.')
if not any(keep):
    raise RuntimeError(
        'No question is usable in both arms. Common causes: a KB is not synced, the '
        'generation model is not enabled in this region, or requests are being throttled.')

_kept_rows = [r for r, k in zip(EVAL_SET, keep) if k]

def score_arm(arm_name, answers, contexts_list):
    rows = [
        {'question': r['question'], 'answer': a, 'contexts': c,
         'ground_truth': r['ground_truth'], 'category': r['category']}
        for r, a, c, k in zip(EVAL_SET, answers, contexts_list, keep) if k
    ]
    ds = Dataset.from_dict({
        'question':     [r['question'] for r in rows],
        'answer':       [r['answer'] for r in rows],
        'contexts':     [r['contexts'] for r in rows],      # list[str] per row
        'ground_truth': [r['ground_truth'] for r in rows],  # single string per row
    })
    res = evaluate(dataset=ds, metrics=METRICS, llm=judge_llm,
                   embeddings=embeddings, run_config=run_cfg)
    df = res.to_pandas()
    df['arm'] = arm_name
    df['category'] = [r['category'] for r in rows]
    return df

print('Scoring Arm A (Retrieve + Converse) ...')
df_a = score_arm('A: Retrieve', arm_a_answers, arm_a_contexts)
print('Scoring Arm B (AgenticRetrieveStream) ...')
df_b = score_arm('B: AgenticRetrieveStream', arm_b_answers, arm_b_contexts)

# NaN check. RAGAS records a throttled or unparseable judge call as NaN, not an error,
# so a "successful" run can still hide missing scores. Surface them, and stop under
# fail_on_error so they are not silently averaged away.
_nan_a = int(df_a[[c for c in METRIC_NAMES if c in df_a]].isna().sum().sum())
_nan_b = int(df_b[[c for c in METRIC_NAMES if c in df_b]].isna().sum().sum())
if _nan_a or _nan_b:
    print(f'NaN metric cells - Arm A: {_nan_a}, Arm B: {_nan_b} (throttling or judge parse failures).')
    if fail_on_error:
        raise RuntimeError(
            'RAGAS returned NaN metric(s). Stopping (fail_on_error=True) rather than '
            'averaging over gaps. Re-run (often transient throttling) or lower '
            'RunConfig max_workers, or set fail_on_error=False to accept the gaps.')
print('Done.')

## Step 9: Compare per category and per question

We report **per category**, never a single blended average (a blend would hide that the win is concentrated in the cross-KB group). The discriminating metrics are `context_recall` and `answer_correctness`: on a cross-KB question the single-KB baseline retrieves only half the evidence, so both drop. `faithfulness` may stay high for both arms: a baseline that honestly says "the document doesn't cover tornadoes" is faithful, just incomplete, so it is not the headline metric.

In [ ]:
import os

combined = pd.concat([df_a, df_b], ignore_index=True)
metric_cols = [c for c in METRIC_NAMES if c in combined.columns]

# Honest run summary: state what THIS run actually evaluated, so nothing downstream
# (including the written Summary) is read as a fixed "two KBs, nine questions" claim.
from collections import Counter as _Counter
_cats = _Counter(r['category'] for r in _kept_rows)
print('=== Run summary ===')
print(f'  Mode: {"two-KB comparison" if two_kb_mode else "single-KB (degraded)"}')
print(f'  Questions scored (both arms): {len(_kept_rows)}  by category: {dict(_cats)}')
print(f'  KBs used: finance={kb_finance_id or "-"}, weather={kb_weather_id or "-"}')

# NaN honesty: throttling / parse failures show up as NaN, not errors. In best-effort
# mode (fail_on_error=False) these survive to here, so track which metric cells are NaN
# and refuse to declare a winner from a category that has any.
nan_by_cat = {}
nan_lines = []
for c in metric_cols:
    n = int(combined[c].isna().sum())
    if n:
        nan_lines.append(f'  {c}: {n}/{len(combined)} rows NaN')
if nan_lines:
    print('NaN counts (NaN means the metric could not be computed for that row):')
    print('\n'.join(nan_lines))

# Per-category, per-arm averages.
per_cat = combined.groupby(['category', 'arm'])[metric_cols].mean().round(3)
print('\n=== Average scores by category and arm ===')
print(per_cat)

# Winner per category on the two discriminating metrics. If either arm has a NaN in the
# discriminating metrics for a category, we do not declare a winner there: a mean() over
# NaN silently drops rows and would make one arm look better than it earned.
print('\n=== Winner per category (context_recall + answer_correctness) ===')
disc = [c for c in ['context_recall', 'answer_correctness'] if c in combined.columns]
for cat in ['finance-only', 'tornado-only', 'cross-kb']:
    sub = combined[combined['category'] == cat]
    if sub.empty:
        continue
    a = sub[sub['arm'] == 'A: Retrieve'][disc]
    b = sub[sub['arm'] == 'B: AgenticRetrieveStream'][disc]
    if a[disc].isna().any().any() or b[disc].isna().any().any():
        print(f'  {cat:12s}  -> inconclusive (NaN in a discriminating metric; re-run)')
        continue
    a_score, b_score = a.mean().mean(), b.mean().mean()
    if abs(a_score - b_score) < 0.05:
        verdict = 'tie'
    elif b_score > a_score:
        verdict = 'Arm B ahead'
    else:
        verdict = 'Arm A ahead'
    print(f'  {cat:12s}  A={a_score:.3f}  B={b_score:.3f}  ->  {verdict}')

# Save outputs. Serialize the list-valued `contexts` column as JSON first: pandas would
# otherwise write it as a Python repr (['a' 'b']) that a generic CSV reader restores as a
# plain string, losing the chunk boundaries. json.dumps round-trips cleanly with
# json.loads (or pd.read_csv followed by json.loads on the column).
os.makedirs('evaluation_output', exist_ok=True)
to_save = combined.copy()
if 'contexts' in to_save.columns:
    to_save['contexts'] = to_save['contexts'].apply(lambda v: json.dumps(list(v)))
to_save.to_csv('evaluation_output/ragas_two_arm_scores.csv', index=False)
per_cat.to_csv('evaluation_output/ragas_two_arm_by_category.csv')
print('\nSaved evaluation_output/ragas_two_arm_scores.csv and ragas_two_arm_by_category.csv')

# Per-question view.
show_cols = ['category', 'arm', 'question'] + metric_cols
combined[show_cols]

## Step 10: Robustness, testing for a chunk-budget artifact

Arm B sees up to `2 × agentic_max_results` chunks (one batch per KB); Arm A sees `retrieve_num_results` from one KB. Someone could argue Arm B only wins because it had more context. To test that, we re-run Arm A on the cross-KB questions with `robustness_n` (default 10) chunks from its single KB and compare.

If the extra chunks barely change Arm A's cross-KB recall and it stays well below Arm B, the gap is structural (more chunks from one KB still hold none of the other KB's facts) rather than a budget effect. If instead the scores shift meaningfully, that conclusion does not hold. The cell below checks the actual numbers and prints whichever applies, with a small-sample caveat (only three cross-KB questions).

In [ ]:
if two_kb_mode:
    cross = [(i, r) for i, r in enumerate(EVAL_SET) if r['category'] == 'cross-kb']
    if cross:
        rob_answers, rob_contexts, rob_errors = [], [], 0
        for i, row in cross:
            try:
                ans, ctx = arm_a_answer_and_contexts(row['question'], ORACLE[row['oracle_kb']], n=robustness_n)
            except Exception as e:
                print(f'  ERROR: {type(e).__name__}: {e}')
                rob_errors += 1
                ans, ctx = '', []
            rob_answers.append(ans)
            rob_contexts.append(ctx)
            print(f"  cross-kb (n={robustness_n}) chunks={len(ctx)}")
            time.sleep(2)
        if fail_on_error and rob_errors:
            raise RuntimeError(
                f'{rob_errors} robustness re-run(s) failed. Stopping (fail_on_error=True) '
                'so the robustness claim is not made on partial data.')

        rob_rows = [
            {'question': r['question'], 'answer': a, 'contexts': c, 'ground_truth': r['ground_truth']}
            for (i, r), a, c in zip(cross, rob_answers, rob_contexts) if a and a.strip() and c
        ]
        if rob_rows:
            rob_ds = Dataset.from_dict({
                'question':     [r['question'] for r in rob_rows],
                'answer':       [r['answer'] for r in rob_rows],
                'contexts':     [r['contexts'] for r in rob_rows],
                'ground_truth': [r['ground_truth'] for r in rob_rows],
            })
            rob_res = evaluate(dataset=rob_ds, metrics=METRICS, llm=judge_llm,
                               embeddings=embeddings, run_config=run_cfg).to_pandas()
            if fail_on_error and rob_res[[c for c in METRIC_NAMES if c in rob_res]].isna().any().any():
                raise RuntimeError('Robustness re-run produced NaN metric(s). Stopping '
                                   '(fail_on_error=True); re-run or lower RunConfig max_workers.')
            a_cross = df_a[df_a['category'] == 'cross-kb']
            b_cross = df_b[df_b['category'] == 'cross-kb']
            a_recall_5  = a_cross['context_recall'].mean()
            a_recall_10 = rob_res['context_recall'].mean()
            b_recall    = b_cross['context_recall'].mean()
            print('\ncross-KB, context_recall + answer_correctness:')
            print(f"  Arm A n={retrieve_num_results:<2d}: recall={a_recall_5:.3f}  correctness={a_cross['answer_correctness'].mean():.3f}")
            print(f"  Arm A n={robustness_n:<2d}: recall={a_recall_10:.3f}  correctness={rob_res['answer_correctness'].mean():.3f}")
            print(f"  Arm B      : recall={b_recall:.3f}  correctness={b_cross['answer_correctness'].mean():.3f}")

            # State the conclusion only if the data supports it: doubling Arm A's budget
            # should barely change its recall AND leave it well below Arm B. Otherwise
            # report what actually happened instead of asserting "structural".
            moved = abs(a_recall_10 - a_recall_5)
            still_below = (b_recall - a_recall_10) > 0.15
            if moved < 0.1 and still_below:
                print(f'\nDoubling Arm A budget moved recall by only {moved:.3f} and it stays '
                      f'{b_recall - a_recall_10:.3f} below Arm B: the gap is structural, not a budget effect.')
            else:
                print(f'\nArm A recall moved {moved:.3f} when the budget doubled (gap to Arm B now '
                      f'{b_recall - a_recall_10:.3f}). With N=3 cross-KB questions this is noisy; '
                      're-run or add questions before drawing a budget-vs-structural conclusion.')
    else:
        print('No cross-KB questions in the trimmed set.')
else:
    print('Single-KB mode: robustness check skipped.')

## Step 11: How to read these results honestly

- **The clear win is on cross-KB questions, in `context_recall` and `answer_correctness`.** That's where a single-KB baseline is structurally unable to gather all the evidence. `Retrieve`'s `answer_relevancy` also collapses there, because a faithful baseline correctly says it can't answer the half its one KB doesn't hold. Don't headline `faithfulness`: an incomplete-but-honest answer still scores well on it.
- **Single-KB groups are the control, and they behave in two ways.** A simple single-fact question ties (both arms see the same chunk). A multi-part question can still favor Arm B, because it splits the question into sub-queries and retrieves for each, while a single broad `Retrieve` can miss one part. That is a real, separate advantage of agentic retrieval (query decomposition), not multi-KB reach, and the notebook reports it rather than hiding it.
- **The baseline is deliberately generous.** Oracle routing picks the best single KB per question, better than a real user gets. It still can't answer cross-KB questions, which is the point.
- **Two asymmetries, both disclosed.** (1) Arm A uses a hand-written generation prompt; Arm B uses the service's internal one. A hand-tuned prompt on a controlled model, if anything, helps Arm A. (2) Arm B sees more chunks; Step 10 shows that isn't what drives the gap.
- **Small N, non-deterministic.** Nine questions is a demo, not a benchmark. Agentic planning is non-deterministic, so scores vary run to run. Scale the question set for real conclusions.
- **The data is synthetic, and the finance doc has internal contradictions.** The Octank 10-K is model-generated and reports *two* different year-end-2021 cash figures ($12,567,000 in the notes vs $480 million in the cash-flow section), and "risk" is classified two ways (market/credit/liquidity vs AFS/HTM/trading). Both variants can appear in retrieved context, so a lower `answer_correctness` sometimes reflects which valid passage an arm surfaced, not a retrieval failure. The tornado report (a real CRS document) has no such conflicts. Treat the scores as illustrative, and prefer cleaner source data for a real evaluation.
- **`answer_correctness` is judge-scored and noisy.** It mixes semantic similarity with an LLM's factual-overlap call, so small differences between arms are within noise; lean on `context_recall` for the cross-KB conclusion.
- **Both arms are deduplicated the same way, and neither follows pagination.** Contexts are de-duplicated by normalized text for both arms, so `context_precision` isn't skewed by one arm repeating a passage. Each arm also takes a single page of results (`numberOfResults`/`maxNumberOfResults` bounds the budget); we don't follow `nextToken`, because paging past that cap would break the equal-budget control. Scale those two parameters, not paging, for a larger evaluation.
- **Holes in the data stop the run, and the comparison stays paired.** With `fail_on_error=True` (default), a generation error, a dropped row, a failed robustness re-run, or a NaN RAGAS score halts before any conclusion is drawn. In best-effort mode (`False`) both arms are scored on the *same* shared subset of questions (a question missing in either arm is dropped from both), so the arms are never compared on different question sets. The printed run summary states how many questions were actually scored.

## Step 12: Cleanup

The cell below deletes **only** what this notebook created, tracked in the `created` record: the two KBs (when `create_kbs` built them), and the bucket plus the two objects it uploaded. It never touches a KB you passed in, and it will not empty or delete a bucket it did not create, so a name collision cannot wipe unrelated data. It reports each deletion honestly and does not claim success for a step that failed. Set `cleanup = True` in the parameters cell, or run the cell manually when you're done.

In [ ]:
# Deletes ONLY what this notebook created, tracked in `created`. We never touch a KB
# you passed in or a bucket we did not make. We also delete only the two objects we
# uploaded and then the (now-empty) bucket, rather than emptying the whole bucket, so
# a name collision could not wipe unrelated data. Deletions are reported honestly:
# we do not claim success on a step that raised.
_created = 'created' in dir() and (created.get('bucket') or created.get('kb_fin') or created.get('kb_wea'))

if not cleanup:
    print('Skipping cleanup. Set cleanup=True to delete resources this notebook created.')
elif not _created:
    print('cleanup=True but this notebook created nothing (KBs were passed in). Nothing deleted.')
else:
    # KBs: delete via the object when we have it, without touching S3 here.
    if created.get('kb_fin') and kb_fin is not None:
        kb_fin.delete_kb(delete_iam=True, delete_s3_bucket=False)
    if created.get('kb_wea') and kb_wea is not None:
        kb_wea.delete_kb(delete_iam=True, delete_s3_bucket=False)

    # Bucket: only if WE created it. Remove just our two keys (and any versions), then
    # the empty bucket. Report exactly what happened.
    b = created.get('bucket')
    if b:
        try:
            s3res = boto3.resource('s3', region_name=region)
            bucket = s3res.Bucket(b)
            for key in created.get('keys', []):
                bucket.objects.filter(Prefix=key).delete()
                bucket.object_versions.filter(Prefix=key).delete()
            bucket.delete()
            print(f'Deleted bucket {b} and its {len(created.get("keys", []))} object(s).')
        except Exception as e:
            print(f'Could NOT fully delete bucket {b}: {type(e).__name__}: {e}')
            print('  The bucket may not be empty (other keys present) or you may lack '
                  'permission. Inspect and delete it manually.')
    print('Cleanup finished (only notebook-created resources were targeted).')

## Summary

This notebook put two Managed-KB retrieval methods head to head and scored both with RAGAS:

1. Built (or reused) knowledge bases holding different documents (finance and weather).
2. **Arm A**: `Retrieve` from one KB, then `Converse` to generate the answer.
3. **Arm B**: `AgenticRetrieveStream` over the available KB(s) with built-in generation.
4. Scored both arms on the **same** question set and tagged each row by category, adding `answer_correctness`.
5. Compared per category.

The exact number of KBs and questions depends on how you ran it: the full two-KB run covers nine questions across three categories (finance-only, tornado-only, cross-KB), while single-KB mode runs only that KB's questions and skips the cross-KB comparison. The Step 9 output and the printed run summary reflect what this run actually did.

**The takeaway (full two-KB run).** `AgenticRetrieveStream` beats a single-KB `Retrieve` in two situations: when a question's evidence is split across more than one knowledge base (it can register several), and when a question has multiple parts (it decomposes and retrieves for each). For a simple single-fact lookup against one KB, the two tie. Pick `Retrieve` for simple single-KB lookups you generate yourself; reach for `AgenticRetrieveStream` for multi-KB or multi-part questions.

### Docs

- [RAGAS documentation](https://docs.ragas.io/)
- [AgenticRetrieveStream for Managed KBs](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-test-agentic-retrieve.html)
- [Retrieve API reference](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_agent-runtime_Retrieve.html)
- See `02-bedrock-evaluation-job.ipynb` and `03-agentcore-evaluation-for-managed-kb.ipynb` for the other two evaluation approaches
